In [7]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import pyFBS
from pyFBS.utility import *
import pickle
import pyvista as pv

In [10]:
import pyvista as pv
import numpy as np
import numpy as np

import cv2

In [11]:
import warnings
warnings.filterwarnings("ignore")

In [62]:
points = np.array([[0.5,0.5,0.5],
                   [0,0.8,0.8],
                   [0.8,0,0.8],
                   [0.8,0.8,0]])




def unit_vector(vector):
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))

class Accelerometer():
    def __init__(self,p,N,mesh = None):

        # accelerometer
        self.box = pv.Box()
        self.box.translate ([1,1,1])
        self.box.points /= 2

        p.add_mesh(self.box,opacity = 0.3, show_edges  = True, color = "#8c8c8c",reset_camera=False)

        ray_x = pv.Line([0,0,0], [1,0,0])
        p.add_mesh(ray_x, color="r", line_width=3,reset_camera=False)
        ray_y = pv.Line([0,0,0], [0,1,0])
        p.add_mesh(ray_y, color="g", line_width=3,reset_camera=False)
        ray_z = pv.Line([0,0,0], [0,0,1])
        p.add_mesh(ray_z, color="b", line_width=3,reset_camera=False)

        self.accelerometer = [self.box,ray_x,ray_y,ray_z]
        self.N = int(N*4)
        self.mesh = mesh
        
        self.mesh.compute_normals(auto_orient_normals = True,inplace = True)
        
        self.local_orientation = np.asarray([[1, 0, 0],
                                        [0, 1, 0],
                                        [0, 0, 1],
                                        [-1, 0, 0],
                                        [0, -1, 0],
                                        [0, 0, -1]]).T
        
    def callback(self,point, i):
        print(i,point)
        # 3D translation in space
        if i == 0:
            self.translate(point,snap = True)
                
        


    def translate(self,point,snap = False):
        
        
        point1 = point + np.asarray([0,0,4])
        point2 = point + np.asarray([0,0,-4])
        
        points, ind = self.mesh.ray_trace(point1,point2)
        
        
        
        if points.size != 0:
            

            
            list_ind = []
            
            # go through all the intersections
            for i in range(len(points)):
                _point = points[i]
                p1 = _point
                p2 = point
                gg  = np.sqrt( ((p1[0]-p2[0])**2)+((p1[1]-p2[1])**2)+((p1[2]-p2[2])**2) )
                
                list_ind.append(gg)
            
            # find the closest to the acc center
            _sel = np.argmin(list_ind)
                
            
            # Find the nearest normal
            v2 = self.mesh.cell_normals[ind[_sel]]
            th = []
            for _loc in self.local_orientation.T:
                th.append(angle_between(_loc, v2))
                
            
            closest_orient = self.local_orientation.T[np.argmin(th)]
            print("orient",closest_orient)
            
            #print(points,ind)
            
            
            # find orientation between acc orientation and cell normal and then push acc 0.5 away from the normal
            f = self.mesh.cell_normals[ind[_sel]]
            t = closest_orient + np.random.random(3)/1e10  # just so that the math works
            
            v = np.cross(f, t)
            u = v/np.linalg.norm(v)
            c = np.dot(f, t)
            h = (1 - c)/(1 - c**2)

            vx, vy, vz = v
            rot =[[c + h*vx**2, h*vx*vy - vz, h*vx*vz + vy],
                  [h*vx*vy+vz, c+h*vy**2, h*vy*vz-vx],
                  [h*vx*vz - vy, h*vy*vz + vx, c+h*vz**2]]
            
            
            
            # rotate the acc accordingly 
            
        
            # PUSH! 
            R = rot       
            sum_plus = R@np.asarray([0,0,0.5])
            
            # simply push in normal direction 
            point = points[_sel] + t/2  #+ sum_plus

                
        else:
            print(points,ind)

            


        _new = point - self.box.center_of_mass() #+ [0.5, 0.5, 0.5]
        
        
        if snap:
            for item in self.accelerometer:
                item.translate(_new) 
                
            p.sphere_widgets[self.N+0].SetCenter(point)

            for i in range(3):
                p.sphere_widgets[self.N +1+ i].SetCenter(_new + np.asarray(p.sphere_widgets[self.N  +1+ i].GetCenter()))
        else:
            for item in self.accelerometer:
                item.translate(_new)
            for i in range(4):
                p.sphere_widgets[self.N  + i].SetCenter(_new + np.asarray(p.sphere_widgets[self.N + i].GetCenter()))

In [63]:
view3D = pyFBS.view3D(show_origin = False)

In [64]:
stl = pyFBS.example_lab_testbench["STL"]["B"]

mesh = pv.PolyData(stl)
mesh.points /= 10

view3D.plot.add_mesh(mesh,name = "ts",color = "#D3D3D3")
view3D.plot.add_mesh(mesh,name = "ts_p",color = "b",style = "points")

(vtkRenderingOpenGL2Python.vtkOpenGLActor)000002834D5A3F48

In [59]:
p = view3D.plot
#p.background_color = "#D4D4D4"

def add_accelerometer(point):
    try:
        i = int(len(view3D.plot.sphere_widgets)/4)
    except:
        i = 0
    _gg = Accelerometer(view3D.plot,i,mesh = mesh)
    view3D.plot.add_sphere_widget(_gg.callback, center=points, color = ["k","r","g","b"],radius = 0.1)
    
    _gg.translate(point)

In [60]:
view3D.plot.enable_point_picking(callback = add_accelerometer,color = "r",show_message="")
view3D.plot.add_text("Press P too add accelerometer", font_size = 12,color = "k")

(vtkRenderingAnnotationPython.vtkCornerAnnotation)000002834D5C04C8

3 (0.8, 0.8, 0.0)
orient [ 0  0 -1]


In [29]:
f = np.asarray([0.0000,0,1]).T + np.random.random(3)/1e10
t = np.asarray([0.0000,0,-1]).T + np.random.random(3)/1e10

import numpy as np
v = np.cross(f, t)
u = v/np.linalg.norm(v)
c = np.dot(f, t)
h = (1 - c)/(1 - c**2)

vx, vy, vz = v
rot =[[c + h*vx**2, h*vx*vy - vz, h*vx*vz + vy],
      [h*vx*vy+vz, c+h*vy**2, h*vy*vz-vx],
      [h*vx*vz - vy, h*vy*vz + vx, c+h*vz**2]]

rot@f

array([ 6.01102344e-11,  5.04171131e-11, -1.00000000e+00])

3 (0.8, 0.8, 0.0)
[[ 1.6841837 16.018244   1.7      ]
 [ 1.6841837 16.018244  -0.3      ]] [1197  878]
3 (0.8, 0.8, 0.0)
[[27.874588 28.252289  2.7     ]
 [27.874588 28.252289  1.7     ]
 [27.874588 28.252289 -0.3     ]
 [27.874588 28.252289 -1.3     ]] [1619 1704 1897 2417]


In [112]:
import numpy as np

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::

            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))

In [137]:
angle = np.asarray([0,10,0])
R = eulerAnglesToRotationMatrix(angle*np.pi/180)

In [141]:
gg = R@np.asarray([0,0,1])*10

np.sqrt(gg[0]**2+gg[1]**2+gg[2]**2)

9.999999999999998

In [110]:
local_orientation

array([[ 1,  0,  0, -1,  0,  0],
       [ 0,  1,  0,  0, -1,  0],
       [ 0,  0,  1,  0,  0, -1]])

In [109]:
local_orientation = np.asarray([[1, 0, 0],
                                        [0, 1, 0],
                                        [0, 0, 1],
                                        [-1, 0, 0],
                                        [0, -1, 0],
                                        [0, 0, -1],]).T

In [118]:
v2 = [0,0,1]

th = []
for _loc in local_orientation.T:
    th.append(angle_between(_loc, v2))
print(th)
np.argmin(th)

[1.5707963267948966, 1.5707963267948966, 0.0, 1.5707963267948966, 1.5707963267948966, 3.141592653589793]


2

In [111]:
v1 = (0,1,0)
v2 = (0,1,0)
theta = angle_between(v1, v2)

In [15]:
orient = np.asarray([[1,0,0],
            [0,1,0],
            [0,0,1]])

In [16]:
R@orient

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [17]:
from scipy.linalg import expm, norm
from numpy import cross, eye


def M(axis, theta):
    """
    Euler-Rodrigues formula
    """
    t = expm(cross(eye(3), axis/norm(axis)*(theta)))
    #print(theta)
     
    return t

In [18]:
axis = v1
v1 = (0,1,0)
v2 = (0,1,0)
theta = angle_between(v1, v2)

M(axis, theta)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [25]:
def rotation_matrix(A,B):
# a and b are in the form of numpy array

    ax = A[0]
    ay = A[1]
    az = A[2]

    bx = B[0]
    by = B[1]
    bz = B[2]

    au = A/(np.sqrt(ax*ax + ay*ay + az*az))
    bu = B/(np.sqrt(bx*bx + by*by + bz*bz))

    R=np.array([[bu[0]*au[0], bu[0]*au[1], bu[0]*au[2]], [bu[1]*au[0], bu[1]*au[1], bu[1]*au[2]], [bu[2]*au[0], bu[2]*au[1], bu[2]*au[2]] ])


    return(R)

In [26]:
def rotation_matrix_from_vectors(vec1, vec2):
    """ Find the rotation matrix that aligns vec1 to vec2
    :param vec1: A 3d "source" vector
    :param vec2: A 3d "destination" vector
    :return mat: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
    """
    a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)
    print(a,b)

    if (np.abs(a) == np.abs(b)).all():
        return np.diag([1,1,1])
    else:
        v = np.cross(a, b)
        print(v)
        c = np.dot(a, b)
        print(c)
        s = np.linalg.norm(v)
        kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        print(s)
        rotation_matrix = np.eye(3) + kmat + kmat.dot(kmat) #* ((1 - c) / (s ** 2))

        return rotation_matrix

In [23]:
vec1 = [0, 1, 0]
vec2 = [0, 0, 1]
rotation_matrix_from_vectors(vec1, vec2)

[0. 1. 0.] [0. 0. 1.]
[1. 0. 0.]
0.0
1.0


array([[ 1.,  0.,  0.],
       [ 0.,  0., -1.],
       [ 0.,  1.,  0.]])